# Event hazards and pit loss — SC, VSC, red rates with Wilson bands (E-03, E-05)

Rates are per-race Bernoulli over every race 2022 through the 2026 calibration
rounds, wet races included: interventions cluster in bad weather and excluding them
would understate the hazard. The band is the Wilson score interval — at five races a
circuit, a wide band is the honest answer, and at 0-of-n it still gives a usable
upper bound. VSC and SC are classified separately throughout (E-03): "12" is green
then yellow, never a safety car.

Pit loss per circuit comes from OpenF1 pit-lane durations (2023 onward): the mean and
sample sigma that seed the E-05 FIFO service-time draws.

In [ ]:
import numpy as np
import pandas as pd
from senna import config
from senna.data.db import database_url, engine_for
from senna.physics.fits import frames, plots
from senna.physics.fits.gate import assert_quality_reports

fits_cfg = config.load("fits")
ingest_cfg = config.load("ingest")
CALIB = config.get(config.load("data_split"), "calibration_rounds")
SLICKS = ("SOFT", "MEDIUM", "HARD")

assert_quality_reports(
    config.get(ingest_cfg, "seasons"),
    config.REPO_ROOT / config.get(ingest_cfg, "quality.report_dir"),
)
engine = engine_for(database_url())
conn = await engine.connect()
index = await frames.circuit_index(conn, CALIB)
index[["slug", "era"]].drop_duplicates().groupby("era").size()

In [ ]:
from senna.physics.fits.hazards import per_lap_hazard, per_race_events, wilson_band

WILSON_Z = config.get(fits_cfg, "hazards.wilson_z")

hazard_rows = []
for slug in sorted(index["slug"].unique()):
    event_name = index.loc[index["slug"] == slug, "event_name"].iloc[0]
    laps = await frames.hazard_laps(conn, event_name, CALIB)
    if laps.empty:
        continue
    events = per_race_events(laps)
    race_laps = float(laps.groupby("session_id")["lap_number"].max().median())
    for kind in ("sc", "vsc", "red"):
        n_with = int(events[kind].sum())
        low, high = wilson_band(n_with, len(events), z=WILSON_Z)
        hazard_rows.append(
            {
                "circuit": slug,
                "kind": kind,
                "n_races": len(events),
                "n_with": n_with,
                "p_per_race": round(events[kind].mean(), 3),
                "band_low": round(low, 3),
                "band_high": round(high, 3),
                "lambda_per_lap": round(per_lap_hazard(events[kind].mean(), race_laps), 5),
            }
        )
hazards_table = pd.DataFrame(hazard_rows)
hazards_table.head(12)

In [ ]:
for kind in ("sc", "vsc", "red"):
    chart = hazards_table[hazards_table["kind"] == kind].set_index("circuit")
    fig = plots.hazard_band_chart(chart, kind)
    print(plots.save_figure(fig, "hazards", f"{kind}_bands"))

In [ ]:
from senna.physics.fits.hazards import status_multipliers

for era in sorted(index["era"].unique()):
    era_laps = await frames.status_laps(conn, era, CALIB)
    print(f"era {era}: {status_multipliers(era_laps)}")

The 5.5.3 `status_mult` values: SC laps run at roughly 1.4x the green median, VSC
laps at roughly 1.2x, in both eras — pooled across circuits because single-circuit
SC lap counts are tiny.

In [ ]:
mismatches = []
for slug in sorted(index["slug"].unique()):
    event_name = index.loc[index["slug"] == slug, "event_name"].iloc[0]
    laps = await frames.hazard_laps(conn, event_name, CALIB)
    messages = await frames.race_control_messages(conn, event_name, CALIB)
    if laps.empty or messages.empty:
        continue
    status_sc = per_race_events(laps)["sc"]
    text = messages.assign(
        says_sc=messages["message"].str.upper().str.contains("SAFETY CAR", na=False)
        & ~messages["message"].str.upper().str.contains("VIRTUAL", na=False)
    )
    message_sc = text.groupby("session_id")["says_sc"].any()
    joined = pd.concat([status_sc, message_sc], axis=1, keys=["track_status", "race_control"])
    for session_id, row in joined.dropna().iterrows():
        if row["track_status"] != row["race_control"]:
            mismatches.append({"circuit": slug, "session_id": session_id, **row})
pd.DataFrame(mismatches) if mismatches else "no SC mismatches between track_status and race control"

Cross-check of the track_status digit parsing against race-control text. Deployment
messages without a status-4 lap (or vice versa) usually mean an SC deployed and
withdrawn inside one lap; track_status is the per-lap exposure record, so it wins.

In [ ]:
from senna.physics.fits.pit_loss import fit_pit_loss

MAX_S = config.get(fits_cfg, "pit_loss.max_duration_s")

pit_rows, duration_samples = [], {}
for slug in sorted(index["slug"].unique()):
    event_name = index.loc[index["slug"] == slug, "event_name"].iloc[0]
    stops = await frames.pit_stops(conn, event_name, CALIB)
    if stops.empty:
        continue
    fit = fit_pit_loss(stops, max_duration_s=MAX_S)
    duration_samples[slug] = stops["pit_duration_s"].dropna()
    pit_rows.append(
        {
            "circuit": slug,
            "mean_s": round(fit.mean_s, 2),
            "sigma_s": round(fit.sigma_s, 2),
            "n_stops": fit.n_stops,
            "n_excluded": fit.n_excluded,
        }
    )
pd.DataFrame(pit_rows)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, slug in zip(axes, ["monaco_grand_prix", "italian_grand_prix", "british_grand_prix"]):
    ax.hist(duration_samples[slug].clip(upper=MAX_S), bins=30)
    ax.set_title(slug)
    ax.set_xlabel("pit duration (s)")
print(plots.save_figure(fig, "pit_loss", "duration_histograms"))

In [ ]:
await conn.close()
await engine.dispose()